In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/Vehicle_Damage_Detection/models/classification", exist_ok=True)
os.makedirs("/content/drive/MyDrive/Vehicle_Damage_Detection/runs", exist_ok=True)


Mounted at /content/drive


In [3]:
!find /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed -maxdepth 2 -name "*archive4*" | head -n 50


/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via.zip


In [5]:
from pathlib import Path
import zipfile

zip_path = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via.zip")
base = zip_path.parent
out_dir = base / "archive4_via"   # folder name after extract

print("Zip exists:", zip_path.exists())

if zip_path.exists() and not out_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(base)

print("Extracted folder exists:", out_dir.exists(), "->", out_dir)


Zip exists: True
Extracted folder exists: True -> /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via


In [8]:
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via")

print("Exists:", DATA.exists())
print("Has train:", (DATA/"train").exists())
print("Has val  :", (DATA/"val").exists())

if (DATA/"train").exists():
    classes = sorted([p.name for p in (DATA/"train").iterdir() if p.is_dir()])
    print("Train classes:", classes)
    for c in classes:
        n = len(list((DATA/"train"/c).glob("*")))
        print(f"  {c}: {n}")


Exists: True
Has train: False
Has val  : False


In [9]:
from pathlib import Path

DATA = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via")

for p in DATA.iterdir():
    print(p.name)


0Train_via_annos.json
images


In [10]:
import json
from pathlib import Path

json_path = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via/0Train_via_annos.json")
data = json.loads(json_path.read_text())

print("Top-level type:", type(data))
# VIA sometimes stores in _via_img_metadata or directly as dict
if isinstance(data, dict) and "_via_img_metadata" in data:
    meta = data["_via_img_metadata"]
else:
    meta = data

# get first item
first_key = next(iter(meta))
first_item = meta[first_key]

print("First item keys:", first_item.keys())
regions = first_item.get("regions", [])
print("Regions count:", len(regions))

if regions:
    r0 = regions[0]
    sa = r0.get("shape_attributes", {})
    ra = r0.get("region_attributes", {})
    print("Shape name:", sa.get("name"))
    print("Shape attributes sample:", sa)
    print("Region attributes sample:", ra)


Top-level type: <class 'dict'>
First item keys: dict_keys(['name', 'regions'])
Regions count: 6
Shape name: None
Shape attributes sample: {}
Region attributes sample: {}


In [11]:
import json
from pathlib import Path

json_path = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via/0Train_via_annos.json")
data = json.loads(json_path.read_text())

# show top-level keys
print("TOP KEYS:", list(data.keys())[:30])

# pick correct metadata section
meta = data.get("_via_img_metadata", data)

first_key = next(iter(meta))
first_item = meta[first_key]

print("\nFIRST ITEM TYPE:", type(first_item))
print("FIRST ITEM KEYS:", list(first_item.keys()))

regions = first_item.get("regions", [])
print("\nREGIONS TYPE:", type(regions), "COUNT:", len(regions))

# print first 2 regions fully (this is the important part)
print("\nREGION[0] =", regions[0])
if len(regions) > 1:
    print("\nREGION[1] =", regions[1])


TOP KEYS: ['01012020_172204image853193.jpg', '01012020_172204image891741.jpg', '01012020_172251image12370.jpg', '01022020_083952image768902.jpg', '01022020_102246image365727.jpg', '01022020_102909image891067.jpg', '01022020_102910image826484.jpg', '01022020_102915image195873.jpg', '01022020_103924image583913.jpg', '01022020_104457image330512.jpg', '01022020_104503image975717.jpg', '01022020_104512image856272.jpg', '01022020_153346image398811.jpg', '01022021_152725image672832.jpg', '01022021_152729image798820.jpg', '01022021_152737image952852.jpg', '01022021_152754image439994.jpg', '01022021_152758image967897.jpg', '01042020_091440image496933.jpg', '01062020_140753image325319.jpg', '01092020_102550image198263.jpg', '01102020_083311image589561.jpg', '01102020_083312image690016.jpg', '01102020_083317image942666.jpg', '01102020_083411image919264.jpg', '01102020_084614image405851.jpg', '01102020_090113image322914.jpg', '01102020_090118image320747.jpg', '01102020_092819image267610.jpg', '011

In [13]:
import json, random, shutil
from pathlib import Path
from PIL import Image, UnidentifiedImageError

# -------------------------
# 1) Paths
# -------------------------
SRC_DIR = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via")
JSON_PATH = SRC_DIR / "0Train_via_annos.json"
IMG_DIR  = SRC_DIR / "images"

OUT_DIR = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_yolo")

img_train = OUT_DIR / "images/train"
img_val   = OUT_DIR / "images/val"
lab_train = OUT_DIR / "labels/train"
lab_val   = OUT_DIR / "labels/val"

for p in [img_train, img_val, lab_train, lab_val]:
    p.mkdir(parents=True, exist_ok=True)

print("JSON exists:", JSON_PATH.exists(), JSON_PATH)
print("images folder exists:", IMG_DIR.exists(), IMG_DIR)

# -------------------------
# 2) Load JSON
# -------------------------
data = json.loads(JSON_PATH.read_text(encoding="utf-8"))
all_files = list(data.keys())

# keep only files that physically exist
existing = [fn for fn in all_files if (IMG_DIR / fn).exists()]
print("Found images referenced in JSON:", len(all_files))
print("Images that exist in folder:", len(existing))

if len(existing) == 0:
    raise Exception("No images found. Check IMG_DIR path or extensions.")

# -------------------------
# 3) Build class list
# -------------------------
class_names = set()
for fn in existing:
    for r in data[fn].get("regions", []):
        if "class" in r:
            class_names.add(r["class"])

class_names = sorted(list(class_names))
class_to_id = {c: i for i, c in enumerate(class_names)}

print("Classes:", class_names)
print("Class mapping:", class_to_id)

# -------------------------
# 4) split train/val
# -------------------------
random.seed(42)
random.shuffle(existing)
split = int(0.8 * len(existing))
train_files = existing[:split]
val_files   = existing[split:]
print("Train:", len(train_files), " Val:", len(val_files))

# -------------------------
# 5) label writer (YOLO-seg)
# -------------------------
def write_label_from_size(w, h, regions, label_path):
    lines = []
    for r in regions:
        xs = r.get("all_x", [])
        ys = r.get("all_y", [])
        cls = r.get("class", None)

        if cls is None or len(xs) < 3 or len(xs) != len(ys):
            continue

        cid = class_to_id[cls]
        pts = []
        for x, y in zip(xs, ys):
            pts.append(x / w)
            pts.append(y / h)

        # clamp to 0..1
        pts = [min(1.0, max(0.0, float(p))) for p in pts]
        lines.append(str(cid) + " " + " ".join(f"{p:.6f}" for p in pts))

    label_path.write_text("\n".join(lines), encoding="utf-8")

# -------------------------
# 6) process split with corrupted-image skip
# -------------------------
def process_split(file_list, out_img_dir, out_lab_dir):
    ok = 0
    bad = 0

    for fn in file_list:
        src_img = IMG_DIR / fn
        dst_img = out_img_dir / fn
        dst_lab = out_lab_dir / (Path(fn).stem + ".txt")

        # Try opening image FIRST (to ensure it is valid)
        try:
            with Image.open(src_img) as im:
                w, h = im.size
                im.verify()  # verify integrity
        except (UnidentifiedImageError, OSError, ValueError):
            bad += 1
            continue

        # Copy image only if valid
        if not dst_img.exists():
            shutil.copy2(src_img, dst_img)

        # Write label using known w,h
        regs = data[fn].get("regions", [])
        write_label_from_size(w, h, regs, dst_lab)

        ok += 1

    return ok, bad

train_ok, train_bad = process_split(train_files, img_train, lab_train)
val_ok,   val_bad   = process_split(val_files, img_val, lab_val)

print("\n DONE")
print("Train valid images:", train_ok, "| skipped corrupted:", train_bad)
print("Val   valid images:", val_ok,   "| skipped corrupted:", val_bad)

# -------------------------
# 7) data.yaml
# -------------------------
yaml_text = f"""path: {OUT_DIR}
train: images/train
val: images/val
names:
"""
for i, c in enumerate(class_names):
    yaml_text += f"  {i}: {c}\n"

(OUT_DIR / "data.yaml").write_text(yaml_text, encoding="utf-8")
print(" data.yaml created:", OUT_DIR / "data.yaml")


JSON exists: True /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via/0Train_via_annos.json
images folder exists: True /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_via/images
Found images referenced in JSON: 11621
Images that exist in folder: 2356
Classes: ['be_den', 'mat_bo_phan', 'mop_lom', 'rach', 'thung', 'tray_son', 'vo_kinh']
Class mapping: {'be_den': 0, 'mat_bo_phan': 1, 'mop_lom': 2, 'rach': 3, 'thung': 4, 'tray_son': 5, 'vo_kinh': 6}
Train: 1884  Val: 472

 DONE
Train valid images: 1884 | skipped corrupted: 0
Val   valid images: 471 | skipped corrupted: 1
 data.yaml created: /content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_yolo/data.yaml


In [14]:
from pathlib import Path

OUT_DIR = Path("/content/drive/MyDrive/Vehicle_Damage_Detection/data/processed/archive4_yolo")
print("Train images:", len(list((OUT_DIR/"images/train").glob("*.jpg"))))
print("Val images  :", len(list((OUT_DIR/"images/val").glob("*.jpg"))))
print("Train labels:", len(list((OUT_DIR/"labels/train").glob("*.txt"))))
print("Val labels  :", len(list((OUT_DIR/"labels/val").glob("*.txt"))))
print("YAML exists :", (OUT_DIR/"data.yaml").exists())


Train images: 1884
Val images  : 472
Train labels: 1884
Val labels  : 471
YAML exists : True


In [4]:
from pathlib import Path
from ultralytics import YOLO

data_yaml = Path(r"C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo\data.yaml")
print("yaml exists:", data_yaml.exists())
print(data_yaml.read_text()[:300])

model = YOLO("yolo11n-seg.pt")
results = model.train(
    data=str(data_yaml),
    imgsz=640,
    epochs=30,
    batch=8,
    name="archive4_yolo11n_seg",
)


yaml exists: True
path: C:/Users/User/Desktop/Vehicle_Damage_Detection/data/processed/archive4_yolo
train: images/train
val: images/val

names:
  0: be_den
  1: mat_bo_phan
  2: mop_lom
  3: rach
  4: thung
  5: tray_son
  6: vo_kinh

Ultralytics 8.4.12  Python-3.11.9 torch-2.10.0+cpu CPU (13th Gen Intel Core i7-13650HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\User\Desktop\Vehicle_Damage_Detection\data\processed\archive4_yolo\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7